In [7]:
import pandas as pd
import numpy as np
import catboost as cb
from geopy.distance import geodesic
from scipy.spatial import KDTree
import ipywidgets as widgets
from IPython.display import display, HTML

import sys
import os
from pathlib import Path

In [10]:
project_root = Path(os.getcwd()).parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Корень проекта добавлен: {project_root}")

from src.my_project.features.constants import KREMLIN, METRO_STATIONS

Корень проекта добавлен: C:\Users\0\PycharmProjects\Gazprom


In [15]:
model = cb.CatBoostRegressor()
MODEL_PATH = project_root / "models" / "catboost_optimized.cbm"
model.load_model(str(MODEL_PATH))

CatBoostRegressor(depth=8, eval_metric='MAPE', iterations=1500, l2_leaf_reg=7, learning_rate=0.03, loss_function='MAE', random_seed=42, use_best_model=True, verbose=200)

In [16]:
def get_prediction(area, lat, lon, floor_cur, floor_tot):
    dist_center = geodesic((lat, lon), KREMLIN).km
    
    metro_coords = list(METRO_STATIONS.values())
    metro_tree = KDTree(metro_coords)
    _, idx = metro_tree.query([lat, lon])
    closest_metro_coords = metro_coords[idx]
    dist_metro = geodesic((lat, lon), closest_metro_coords).km
    
    floor_ratio = floor_cur / floor_tot if floor_tot > 0 else 0
    
    input_df = pd.DataFrame({
        'area_m2': [area],
        'floor_current': [floor_cur],
        'floor_total': [floor_tot],
        'latitude': [lat],
        'longitude': [lon],
        'dist_to_center': [dist_center],
        'dist_to_metro': [dist_metro],
        'floor_ratio': [floor_ratio]
    })
    
    price = model.predict(input_df)[0]
    return price, dist_center, dist_metro

In [17]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Стилизация
style = {'description_width': 'initial'}

area_input = widgets.FloatSlider(value=60, min=10, max=300, step=1, description='Площадь (м²):', style=style)
floor_cur_input = widgets.IntSlider(value=5, min=1, max=50, description='Этаж:', style=style)
floor_tot_input = widgets.IntSlider(value=12, min=1, max=50, description='Всего этажей:', style=style)
lat_input = widgets.FloatText(value=55.7512, description='Широта (Lat):', style=style)
lon_input = widgets.FloatText(value=37.6184, description='Долгота (Lon):', style=style)

output = widgets.Output()

def update_prediction(change):
    with output:
        output.clear_output()
        price, d_center, d_metro = get_prediction(
            area_input.value, lat_input.value, lon_input.value, 
            floor_cur_input.value, floor_tot_input.value
        )
        
        display(HTML(f"""
            <div style="background-color: #f0f2f6; padding: 20px; border-radius: 10px; border-left: 5px solid #2e7d32;">
                <h2 style="margin: 0; color: #1b5e20;">Прогноз стоимости: {price:,.0f} ₽</h2>
                <p style="margin: 5px 0; color: #444;">Цена за м²: <b>{price/area_input.value:,.0f} ₽</b></p>
                <hr>
                <p style="margin: 5px 0;">📍 До центра: {d_center:.2f} км</p>
                <p style="margin: 5px 0;">🚇 До метро: {d_metro:.2f} км</p>
            </div>
        """))

# Подписываем виджеты на изменения
for w in [area_input, floor_cur_input, floor_tot_input, lat_input, lon_input]:
    w.observe(update_prediction, names='value')

# Отображаем всё
display(HTML("<h1>🏠 Оценщик недвижимости (CatBoost)</h1>"))
display(widgets.VBox([area_input, floor_cur_input, floor_tot_input, lat_input, lon_input, output]))

# Первый запуск
update_prediction(None)

In [ ]:
55.760018, 37.831839